# Logistic Regression Refuses to Converge

**Topic:** Newton's method / IRLS &nbsp;·&nbsp; **Difficulty:** Hard

On perfectly separable data the maximum-likelihood logistic fit does not exist.
The likelihood improves as the weights grow, but the weights do not run away to
infinity. Once the fitted probabilities saturate at exactly 0 and 1 the gradient
they contribute is zero, Newton's steps shrink to nothing, and the final
`||w||` lands near 161, not `1e+15`.

Watch it happen, then add the one term that fixes it. A ridge penalty makes the
objective strictly convex and coercive, so a finite optimum exists again.

## The data (created in the next cell)

- `X` `(about 271, 4)`, `y` `(about 271,)`: **linearly separable** by
  construction (rows with `|margin| <= 0.35` are dropped, so the exact count
  varies with the seed).
- No two classes overlap, so any separating hyperplane has zero training error.

## Your task

With **NumPy only**:

1. Run 40 Newton steps on the *unpenalized* log-likelihood. The Hessian is
   `X^T diag(p(1-p)) X`. Add `1e-10 * I` before solving so the linear algebra
   does not raise, and record `norm_unpen` = the final `||w||`.
2. Run 40 Newton steps on the L2-penalized objective with `lam = 1.0`:
   gradient `X^T (p - y) + lam * w`, Hessian `X^T diag(p(1-p)) X + lam * I`.
3. Set `answers` to a dict with `"norm_unpen"` (float), `"w_pen"` `(4,)`,
   `"norm_pen"` (float), and `"train_acc_pen"` (float).

## Evaluation

The penalized fit is compared to the reference to 1e-6. The evaluation cell also
checks the unpenalized norm came out far larger than the penalized one.

In [ ]:
import numpy as np

rng = np.random.default_rng(505)
n, d = 300, 4
w_sep = np.array([1.5, -2.0, 0.7, 1.2])
X = rng.normal(0, 1, (n, d))
margin = X @ w_sep
keep = np.abs(margin) > 0.35            # carve out a clean gap
X, margin = X[keep], margin[keep]
y = (margin > 0).astype(float)
n = len(y)
print(f"X {X.shape}, separable: {bool(np.all((X @ w_sep > 0) == (y > 0)))}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("w_pen", [4], 2.5923643144511135, 0.36159757210814725, 1e-06),
    ("norm_pen", [], 5.392222164216542, -0.9028629746151601, 1e-06),
    ("train_acc_pen", [], 1.0, -0.16743801481449916, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "norm_unpen" not in answers:
    print(f"  {'norm_unpen':<24} MISSING")
    _ok = False
if "norm_unpen" in answers and "norm_pen" in answers:
    _blew = answers["norm_unpen"] > 20 * max(answers["norm_pen"], 1e-9)
    print(f"  {'unpenalized diverges':<24} {'ok' if _blew else 'wrong'}  "
          f"(||w|| {answers['norm_unpen']:.3g} vs {answers['norm_pen']:.3g})")
    _ok = _ok and _blew

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Without a penalty the fitted probabilities saturate at exactly 0 and 1, so
`p(1-p)` is 0.0 for those points and their contribution to the gradient
vanishes. The Hessian keeps its `1e-10 * I` jitter and never goes singular,
but with the gradient gone Newton's steps shrink to nothing and `||w||`
settles near 161 after 40 of them. With `lam * I` added, the Hessian is
bounded below by `lam` regardless of saturation, so Newton converges to the
unique finite minimizer, the same reason
`sklearn`'s `LogisticRegression` regularizes by default and why `C=inf` is not
the safe choice people assume.

```python
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

d = X.shape[1]

w = np.zeros(d)
for _ in range(40):
    p = sigmoid(X @ w)
    g = X.T @ (p - y)
    H = X.T @ (X * (p * (1 - p))[:, None]) + 1e-10 * np.eye(d)
    w = w - np.linalg.solve(H, g)
norm_unpen = float(np.linalg.norm(w))

lam = 1.0
wp = np.zeros(d)
for _ in range(40):
    p = sigmoid(X @ wp)
    g = X.T @ (p - y) + lam * wp
    H = X.T @ (X * (p * (1 - p))[:, None]) + lam * np.eye(d)
    wp = wp - np.linalg.solve(H, g)

answers = {
    "norm_unpen": norm_unpen,
    "w_pen": wp,
    "norm_pen": float(np.linalg.norm(wp)),
    "train_acc_pen": float(((sigmoid(X @ wp) > 0.5).astype(float) == y).mean()),
}
```